# Lezione 11 · L'economia del gesto

Il banco, il magazzino e il carrello: i conti dell'inferenza, la quantizzazione dei quattro pesi, e il conto in euro.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## Il carrello

Una scheda grande di qualche anno fa sposta circa 3.300 miliardi di byte al secondo.

In [2]:
banda = 3300e9
pesi = 70e9 * 2                      # 70 miliardi a 16 bit
controlla("secondi per token", pesi / banda, 0.04, 0.003)
print(f"token al secondo, per una persona sola: {banda / pesi:.0f}")
operazioni_al_secondo = 2 * 70e9 * banda / pesi
print(f"il banco lavora a {operazioni_al_secondo/1e12:.1f} mila miliardi di operazioni al secondo")

✓  secondi per token: calcolato 0.04242, nel libro 0.04
token al secondo, per una persona sola: 24
il banco lavora a 3.3 mila miliardi di operazioni al secondo


## Il quaderno, cioè la KV cache

In [3]:
def byte_per_token(strati, teste_kv, dim_testa, byte=2):
    return 2 * strati * teste_kv * dim_testa * byte

bpt = byte_per_token(80, 8, 128)
controlla("byte per token", bpt, 327_680, 0)
controlla("GB a 128.000 token", bpt * 128_000 / 1e9, 42, 0.1)
controlla("lettura di 10.000 token (mila miliardi di operazioni)", 2 * 70e9 * 10_000 / 1e12, 1400, 0)

✓  byte per token: calcolato 3.277e+05, nel libro 327680
✓  GB a 128.000 token: calcolato 41.94, nel libro 42
✓  lettura di 10.000 token (mila miliardi di operazioni): calcolato 1400, nel libro 1400


## La quantizzazione dei quattro pesi

In [4]:
def quantizza(pesi, bit):
    livelli = 2 ** (bit - 1) - 1          # 127 a 8 bit, 7 a 4 bit
    scala = np.max(np.abs(pesi)) / livelli
    interi = np.round(pesi / scala)
    return interi, interi * scala

pesi = np.array([0.12, -0.48, 0.91, -0.07])
interi, ricostruiti = quantizza(pesi, 8)
assert interi.tolist() == [17, -67, 127, -10]
for r, att in zip(ricostruiti, [0.122, -0.480, 0.910, -0.072]):
    controlla("a 8 bit", r, att, 0.001)

_, r4 = quantizza(pesi, 4)
for r, att in zip(r4, [0.13, -0.52, 0.91, -0.13]):
    controlla("a 4 bit", r, att, 0.005)

_, r4g = quantizza(np.append(pesi, 9.0), 4)
print("con il gigante da 9,0:", np.round(r4g, 2))
assert (r4g[[0, 1, 3]] == 0).all()
controlla("lo 0,91 diventa", r4g[2], 1.29, 0.01)

✓  a 8 bit: calcolato 0.1218, nel libro 0.122
✓  a 8 bit: calcolato -0.4801, nel libro -0.48
✓  a 8 bit: calcolato 0.91, nel libro 0.91
✓  a 8 bit: calcolato -0.07165, nel libro -0.072
✓  a 4 bit: calcolato 0.13, nel libro 0.13
✓  a 4 bit: calcolato -0.52, nel libro -0.52
✓  a 4 bit: calcolato 0.91, nel libro 0.91
✓  a 4 bit: calcolato -0.13, nel libro -0.13
con il gigante da 9,0: [ 0.   -0.    1.29 -0.    9.  ]
✓  lo 0,91 diventa: calcolato 1.286, nel libro 1.29


## Fino a quattro volte, ma solo a pagina corta

In [5]:
kv_lunga = 42e9
print(f"a pagina corta: {140 / 35:.1f} volte più veloce")
print(f"a 128.000 token: {(140e9 + kv_lunga) / (35e9 + kv_lunga):.2f} volte più veloce")

a pagina corta: 4.0 volte più veloce
a 128.000 token: 2.36 volte più veloce


## Il conto in euro

3 € per milione di token letti, 15 € per milione di token scritti; mille domande al giorno da 3.000 token, risposte da 500.

In [6]:
lettura = 1000 * 3000 / 1e6 * 3
scrittura = 1000 * 500 / 1e6 * 15
controlla("lettura al giorno (€)", lettura, 9, 0)
controlla("scrittura al giorno (€)", scrittura, 7.5, 0)
print(f"totale: {lettura + scrittura:.2f} € al giorno, circa {30*(lettura + scrittura):.0f} € al mese, {365*(lettura + scrittura):,.0f} € l'anno")

✓  lettura al giorno (€): calcolato 9, nel libro 9
✓  scrittura al giorno (€): calcolato 7.5, nel libro 7.5
totale: 16.50 € al giorno, circa 495 € al mese, 6,022 € l'anno


## Kata per casa

In [7]:
for formato, byte, atteso in [("fp16", 2, 140), ("int8", 1, 70), ("int4", 0.5, 35)]:
    controlla(f"70 miliardi in {formato} (GB)", 70e9 * byte / 1e9, atteso, 0)
controlla("frazione del calcolo degli esperti (%)", 2 / 8 * 100, 25, 0)

✓  70 miliardi in fp16 (GB): calcolato 140, nel libro 140
✓  70 miliardi in int8 (GB): calcolato 70, nel libro 70
✓  70 miliardi in int4 (GB): calcolato 35, nel libro 35
✓  frazione del calcolo degli esperti (%): calcolato 25, nel libro 25
